In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/nithyasrir/Personal/RAG-workshop/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: IT_Security_Manual.pdf
  ✓ Loaded 3 pages

Processing: HR_Policy_Document.pdf
  ✓ Loaded 3 pages

Processing: University_Handbook.pdf
  ✓ Loaded 3 pages

Processing: Banking_Policy_Document.pdf
  ✓ Loaded 3 pages

Total documents loaded: 12


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/IT_Security_Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'IT_Security_Manual.pdf', 'file_type': 'pdf'}, page_content='IT Security and Compliance Manual\nPassword Policy\nPasswords must be at least 12 characters long.\nPasswords must include uppercase, lowercase, numbers, and symbols.\nPasswords must be changed every 90 days.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source'

In [4]:
### Split documents into chunks using RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

pdf_chunks = text_splitter.split_documents(all_pdf_documents)
print(f"Split {len(all_pdf_documents)} documents into {len(pdf_chunks)} chunks")

Split 12 documents into 12 chunks


In [5]:
chunks = text_splitter.split_documents(all_pdf_documents)
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/IT_Security_Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'IT_Security_Manual.pdf', 'file_type': 'pdf'}, page_content='IT Security and Compliance Manual\nPassword Policy\nPasswords must be at least 12 characters long.\nPasswords must include uppercase, lowercase, numbers, and symbols.\nPasswords must be changed every 90 days.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source'

Embedding and vector DB


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights:   1%|          | 1/103 [00:00<00:00, 4899.89it/s, Materializing param=embeddings.LayerNorm.bias]


Loading weights:   1%|          | 1/103 [00:00<00:00, 2251.37it/s, Materializing param=embeddings.LayerNorm.bias]


Loading weights:   2%|▏         | 2/103 [00:00<00:00, 1175.04it/s, Materializing param=embeddings.LayerNorm.weight]

...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2323.63it/s, Materializing param=pooler.dense.weight]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2316.20it/s, Materializing param=pooler.dense.weight]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2305.64it/s, Materializing param=pooler.dense.weight]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


Vector store

Vector Store

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 12


In [9]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/IT_Security_Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'IT_Security_Manual.pdf', 'file_type': 'pdf'}, page_content='IT Security and Compliance Manual\nPassword Policy\nPasswords must be at least 12 characters long.\nPasswords must include uppercase, lowercase, numbers, and symbols.\nPasswords must be changed every 90 days.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source'

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 12 texts...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]

Generated embeddings with shape: (12, 384)
Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents in collection: 24


In [11]:
vectorstore

Retriever Pipeline From VectorStore